The goal of this notebook is to take the large Transsee csv files and split it into managable chunks. We'll aim for csv files with
500k rows. This assumes that the large csv files

In [13]:
import pandas as pd
import numpy as np
import os

In [14]:
# Here I load the csv files in the current directory. Note that since the large files do not
# upload to github, running this notebook should do nothing.

csv_files = [file for file in os.listdir() if '.csv' == file[-4:]]
file_index = dict()
for file in csv_files:
    file_index[file] = int(file[-5])

csv_files = sorted(csv_files, key=lambda x: file_index[x])

dataframe = [pd.read_csv(csv) for csv in csv_files]

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_41598/3126056355.py:11: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = [pd.read_csv(csv) for csv in csv_files]


Sometimes it can happen that some of the csv files are missing columns. We will take the conservative approach and just put NaNs when there are columns missing.

In [15]:
combined_columns = set()
for df in dataframe:
    combined_columns = combined_columns.union(set(df.columns))

combined_columns = list(combined_columns)
print(combined_columns)

['Destination', 'day of the week', 'Schedule', 'Unnamed: 0', 'Vehicle', 'Riders after stop', 'day', 'Time', 'Gap', 'Headway']


In [16]:
# Add NaNs when a column is missing
for df in dataframe:
    for column in combined_columns:
        if column not in df.columns:
            df[column] = [np.nan for _ in range(len(df))]

merged_df = pd.concat(dataframe, ignore_index=True)

In [17]:
def split_dataframe_into_chunks(df: pd.DataFrame, chunk_size: int):
    return [df[k:k+chunk_size] for k in range(0, len(df), chunk_size)]

In [18]:
df_split = split_dataframe_into_chunks(merged_df, 500000)

In [19]:
data_name = '506_schedule_data'
for idx, df in enumerate(df_split):
    df.to_csv(f'../data/raw_data/schedule_data/{data_name}_{idx}.csv', index=False)